In [1]:
import pandas as pd
import sqlite3

In [2]:
# Conecta ao banco
conn = sqlite3.connect('../dados-publicos/exp_cnpj.db')

# Lê o Excel com os valores de consulta
df_filtro = pd.read_excel('../data/cnpjs_mapeaveis.xlsx')

# Cria uma tabela temporária com os CNPJs do Excel
df_filtro[['cnpj']].to_sql('filtro_cnpj', conn, if_exists='replace', index=False)


5250

In [3]:
sql = """
SELECT
    e.*,
    m.exp_description AS nome_municipio,
    c.exp_description AS nome_cnae
FROM exp_establishment e
JOIN exp_municipality m ON m.exp_code = e.exp_municipality
JOIN exp_cnae c ON c.exp_code = e.exp_primary_cnae
WHERE e.exp_cnpj IN (SELECT cnpj FROM filtro_cnpj)
"""
df_resultado = pd.read_sql(sql, conn)

In [7]:
print(df_resultado.shape)
df_resultado.head(5)

(5230, 33)


,exp_cnpj_base,exp_cnpj_order,exp_cnpj_check_digit,exp_headquarters_branch,exp_trade_name,exp_registration_status,exp_registration_status_date,exp_registration_status_reason,exp_foreign_city_name,exp_country,...,exp_area_code2,exp_phone2,exp_fax_area_code,exp_fax,exp_email,exp_special_status,exp_special_status_date,exp_cnpj,nome_municipio,nome_cnae
0,85318871,0001,00,1,,02,20010324,00,,,...,,,,,,,,85318871000100,BLUMENAU,Comércio a varejo de peças e acessórios usados...
1,04023427,0001,66,1,,02,20040402,00,,,...,19,34063845,19,34076002,FISCAL@DUARTECON.COM.BR,,,04023427000166,AMERICANA,Comércio varejista especializado de equipament...
2,51371425,0001,48,1,ROGER'S,02,20051103,00,,,...,,,,,,,,51371425000148,SUZANO,Comércio varejista de artigos do vestuário e a...
3,68445345,0001,92,1,POLIESPIRAL,02,20050924,00,,,...,,,011,2236261,,,,68445345000192,SAO PAULO,Comércio varejista de equipamentos para escrit...
4,72832686,0001,98,1,HELTER SKELTER ROCK SHOP,02,20051103,00,,,...,,,,,VINILNANET@GMAIL.COM,,,72832686000198,SAO PAULO,"Comércio varejista de discos, CDs, DVDs e fitas"


In [ ]:
conn.execute('DROP TABLE IF EXISTS filtro_cnpj')
conn.close()

In [ ]:
df_merge = pd.merge(df_resultado, df_filtro[['cnpj', 'seller_email', 'seller_celular', 'dominio', 'site']], left_on='exp_cnpj', right_on='cnpj', how='left')

In [13]:
df_merge['endereco_completo'] = df_merge.apply(
        lambda row: f"{row['exp_street_type']} {row['exp_street']}, {row['exp_number']}, {row['exp_neighborhood']}, {row['nome_municipio']} - {row['exp_state']}, {row['exp_zip_code']}"
    if pd.notna(row['exp_street_type']) and pd.notna(row['exp_street']) and pd.notna(row['exp_number']) and pd.notna(row['exp_neighborhood']) and pd.notna(row['nome_municipio']) and pd.notna(row['exp_state']) and pd.notna(row['exp_zip_code'])
    else None,
    axis=1
)

In [9]:
df_resultado['endereco_completo'] = df_resultado.apply(
        lambda row: f"{row['exp_street_type']} {row['exp_street']}, {row['exp_number']}, {row['exp_neighborhood']}, {row['nome_municipio']} - {row['exp_state']}, {row['exp_zip_code']}"
    if pd.notna(row['exp_street_type']) and pd.notna(row['exp_street']) and pd.notna(row['exp_number']) and pd.notna(row['exp_neighborhood']) and pd.notna(row['nome_municipio']) and pd.notna(row['exp_state']) and pd.notna(row['exp_zip_code'])
    else None,
    axis=1
)

In [10]:
df_resultado.to_csv('../saidas/estabelecimentos_mapeaveis.csv', index=False)

In [14]:
df_merge.to_excel('../saidas/consulta_cnpjs_NF_0326.xlsx', index=False)